# Batch 3 — E21, E23, E22 (validation only)

**Runtime:** Runtime → Change runtime type → **L4 GPU**. Then Runtime → Run all; about **2 – 2 h 30 min**. Keep the tab open.
If Colab disconnects, Run all again: finished runs are skipped.

Fixed before any run (`claude/batch3-design-2026-10-16.md`):

| Run | What | Order |
|---|---|---|
| E21 | **Stacked parent**: RevIN off + classification only + stop when every horizon has stalled + best epoch per horizon (E10 + E11b + E19 + E12) | 1st; its first 12 epochs must match E11b exactly (free reproducibility check) |
| E23 | E21 but **RevIN back on, with each input's window mean and log-std fed to the heads** | 2nd |
| E22 | **Walk-forward check of E21**: same 4 windows as E05 / E20 (XGBoost fold files reused from E05) + official window | 3rd (slowest) |

Seeds 20261015/16/17. Outputs in `MyDrive/Senior/results/batch3/`. Test block stays sealed.

In [ ]:
!nvidia-smi -L

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
SENIOR = '/content/drive/MyDrive/Senior'
PRICE_DIR, OUT_DIR = f'{SENIOR}/prices', f'{SENIOR}/results'

In [ ]:
!rm -rf /content/Senior-Project && git clone -q https://github.com/Smith8bit/Senior-Project.git /content/Senior-Project
%cd /content/Senior-Project
!git log --oneline -1
import hashlib, os
assert '--revin-stats' in open('scripts/baselines_deep.py', encoding='utf-8').read(), 'stop: push the latest scripts/baselines_deep.py'
assert 'revin_stats' in open('src/multiscale_patchtst.py', encoding='utf-8').read(), 'stop: push the latest src/multiscale_patchtst.py'
for f in ['scripts/baselines_deep.py', 'src/multiscale_patchtst.py', 'src/stopping.py']:
    print(f, hashlib.sha256(open(f, 'rb').read()).hexdigest()[:12])
FOLDS = [('2022H2', '2022-07-01', '2022-12-31'), ('2023H1', '2023-01-01', '2023-06-30'),
         ('2023H2', '2023-07-01', '2023-12-31'), ('2024H1', '2024-01-01', '2024-06-30')]
need = [f'{OUT_DIR}/batch1/e10_no_revin.json', f'{OUT_DIR}/batch2/e11b_cls_only.json', f'{OUT_DIR}/batch2/e19_every_horizon.json',
        f'{OUT_DIR}/xgb_e03.json', f'{OUT_DIR}/e05/summary_e05.json', f'{OUT_DIR}/batch2/e20_summary_per_horizon.json',
        f'{OUT_DIR}/batch2/e20_summary_shared_epoch.json'] + [f'{OUT_DIR}/e05/{f}_xgb.json' for f, _, _ in FOLDS]
for f in need:
    assert os.path.exists(f), f'stop: missing {f}'
B3 = f'{OUT_DIR}/batch3'
os.makedirs(f'{B3}/e22', exist_ok=True)
print('earlier files found; outputs go to', B3)

In [ ]:
!mkdir -p data/raw/prices/siamchart data/raw/prices/yahoo "$OUT_DIR"
!cp "$PRICE_DIR/set50_union_2019_2025.csv" data/raw/prices/siamchart/
!cp "$PRICE_DIR/yahoo_events_2019_2025.csv" data/raw/prices/yahoo/
# Files must match the PC copies (hashes recorded 2026-10-15; sentiment file as stored in git, LF line endings)
expected = {'data/raw/prices/siamchart/set50_union_2019_2025.csv': '9d7ce4c3e72a974979566dc089d5b8819af54dc24f62b71240f5b4716b556e6f',
            'data/raw/prices/yahoo/yahoo_events_2019_2025.csv': 'd021a58df95fa487f1e49eb50617f6b3187b025b6377235c6180166b1ec84d53',
            'data/derived/sentiment_scores.csv': '7369fd981c4ccffc017d6d5de50ed95c6a4ac4d843bbc6053366029ce614bb1b'}
for path, h in expected.items():
    got = hashlib.sha256(open(path, 'rb').read()).hexdigest()
    print('OK ' if got == h else 'MISMATCH ', path)
    assert got == h, 'stop: a file differs from the PC copy'

In [ ]:
# Pipeline, look-ahead, walk-forward, stop-rule and model tests; the notebook stops here if any fails (about 5 minutes)
import subprocess
for t in ['tests/test_pipeline.py', 'tests/test_no_lookahead.py', 'tests/test_walk_forward.py', 'tests/test_stopping.py',
          'tests/test_multiscale_patchtst.py']:
    r = subprocess.run(['python', t], capture_output=True, text=True)
    print(f'--- {t}'); print('\n'.join(l for l in r.stdout.splitlines() if 'defined on' not in l)[-1500:])
    assert r.returncode == 0 and 'FAIL' not in r.stdout, f'stop: {t} failed'

In [ ]:
# Switch check for E23 (stops on mismatch)
import sys, torch
sys.path.insert(0, 'scripts'); sys.path.insert(0, 'src')
import baselines_deep as BD
def n(m): return sum(p.numel() for p in m.parameters())
BD.OPT.update(revin=True, cls_only=True, label_smoothing=0.0, stop_rule='every', revin_stats=False)
plain = BD.make_model('multiscale_sector', 32, 18, 0.2).eval()
BD.OPT.update(revin_stats=True)
stats = BD.make_model('multiscale_sector', 32, 18, 0.2).eval()
BD.OPT.update(revin=False, revin_stats=False)
parent = BD.make_model('multiscale_sector', 32, 18, 0.2).eval()
BD.OPT.update(revin=True, revin_stats=False)
print('weights: E21 parent (RevIN off)', n(parent), '| RevIN on', n(plain), '| E23 RevIN on + window stats', n(stats))
# trunk input grows by 2 x 32 = 64 values -> 64 x 128 extra weights
assert n(parent) == 472_572 and n(plain) == 472_636 and n(stats) == 472_636 + 64 * 128 == 480_828, 'stop: weight count differs from the design'
torch.manual_seed(0)
x, g = torch.randn(8, 60, 32), torch.randint(0, 19, (8,))
shift = torch.randn(1, 1, 32) * 3              # move every input's level by a constant
with torch.no_grad():
    same = torch.allclose(plain(x, g)['trend_logits'], plain(x + shift, g)['trend_logits'], atol=1e-4)
    moved = not torch.allclose(stats(x, g)['trend_logits'], stats(x + shift, g)['trend_logits'], atol=1e-4)
print('RevIN alone ignores a level shift:', same, '| with window stats the shift is seen:', moved)
assert same and moved, 'stop: window statistics are not reaching the heads'
r = subprocess.run(['python', 'scripts/baselines_deep.py', '--no-revin', '--revin-stats', '--models', 'multiscale_sector'], capture_output=True, text=True)
assert r.returncode != 0 and 'cannot be combined' in r.stderr, 'stop: --revin-stats with --no-revin must be refused'
print('switch check passed')

In [ ]:
# Smoke run (2 epochs, 1 seed, scratch file) with the E23 switches
import json
subprocess.run(['python', 'scripts/baselines_deep.py', '--models', 'multiscale_sector', '--news', 'data/derived/sentiment_scores.csv',
                '--extra-features', 'e03', '--lr', '1e-4', '--dropout', '0.2', '--max-epochs', '2', '--patience', '1',
                '--cls-only', '--stop-rule', 'every', '--revin-stats', '--seeds', '20261015', '--out', '/content/smoke.json'],
               check=True, capture_output=True)
res = json.load(open('/content/smoke.json'))
run = res['models']['multiscale_sector']['runs'][0]
o = res['settings']['options']
assert o['revin'] and o['revin_stats'] and o['cls_only'] and o['stop_rule'] == 'every' and run['params'] == 480_828, 'stop: smoke output wrong'
print('smoke run passed:', run['epochs_run'], 'epochs, best epoch per horizon', run['best_epoch_per_horizon'])

In [ ]:
import json, subprocess, sys
BASE = ['--models', 'multiscale_sector', '--news', 'data/derived/sentiment_scores.csv', '--extra-features', 'e03',
        '--lr', '1e-4', '--dropout', '0.2', '--max-epochs', '30', '--patience', '10', '--cls-only', '--stop-rule', 'every']
E21 = BASE + ['--no-revin']                     # stacked parent
E23 = BASE + ['--revin-stats']                  # RevIN on + window statistics
def done(path):
    try:
        return len(json.load(open(path))['models']['multiscale_sector']['runs']) == 3
    except Exception:
        return False
def run(label, args, out):
    if done(out):
        print(f'=== {label}: already done'); return
    print(f'=== {label}'); sys.stdout.flush()
    subprocess.run(['python', 'scripts/baselines_deep.py', *args, '--out', out], check=True)

# 1. E21: the stacked parent
run('E21 stacked parent', E21, f'{B3}/e21_stacked.json')

In [ ]:
# Reproducibility check (free): E21 trains exactly like E11b until E11b stopped (epoch 12), so those epochs must match.
p = {r['seed']: r for r in json.load(open(f'{OUT_DIR}/batch2/e11b_cls_only.json'))['models']['multiscale_sector']['runs']}
diffs = []
for r in json.load(open(f'{B3}/e21_stacked.json'))['models']['multiscale_sector']['runs']:
    a, b = p[r['seed']]['history'], r['history']
    k = len(a)
    d = max(max(abs(x - y) for x, y in zip(u['es_per_horizon'], v['es_per_horizon'])) for u, v in zip(a, b[:k]))
    diffs.append(d)
    print(f"seed {r['seed']}: E11b stopped at epoch {k}, E21 at epoch {r['epochs_run']}; "
          f"max |difference| over epochs 1-{k}: {d:.4f}; best epoch per horizon {r['best_epoch_per_horizon']}")
print('REPRODUCED EXACTLY' if max(diffs) == 0 else 'NOTE: not bit-identical to batch 2 - tell Claude')

In [ ]:
# 2. E23: RevIN on + window statistics (otherwise identical to E21)
run('E23 RevIN + window stats', E23, f'{B3}/e23_revin_stats.json')

In [ ]:
# 3. E22: walk-forward check of E21 (deep model per window; XGBoost fold files reused from E05)
for label, es, ee in FOLDS:
    run(f'E22 {label}', E21 + ['--eval-start', es, '--eval-end', ee], f'{B3}/e22/{label}_deep.json')

In [ ]:
# Summary 1: does stacking add up? E19, E11b and E21 against the batch-2 parent (E10+E12), per-horizon best epoch
!python scripts/compare_batch.py --per-horizon-best --control-label "E10+E12" --control "$OUT_DIR/batch1/e10_no_revin.json" \
    --run E19 "$OUT_DIR/batch2/e19_every_horizon.json" --run E11b "$OUT_DIR/batch2/e11b_cls_only.json" \
    --run E21 "$OUT_DIR/batch3/e21_stacked.json" --out "$OUT_DIR/batch3/batch3_stacking.json"
# Summary 2: E23 against the new parent E21
!python scripts/compare_batch.py --per-horizon-best --control-label "E21" --control "$OUT_DIR/batch3/e21_stacked.json" \
    --run E23 "$OUT_DIR/batch3/e23_revin_stats.json" --out "$OUT_DIR/batch3/batch3_summary.json"

In [ ]:
# Summary 3: E22 walk-forward, scored both ways
WF = ' '.join(f'--fold {f} "{OUT_DIR}/batch3/e22/{f}_deep.json" "{OUT_DIR}/e05/{f}_xgb.json"' for f, _, _ in FOLDS)
WF += f' --fold official "{OUT_DIR}/batch3/e21_stacked.json" "{OUT_DIR}/xgb_e03.json"'
!python scripts/summarize_walk_forward.py --experiment E22 --per-horizon-best {WF} --out "$OUT_DIR/batch3/e22_summary_per_horizon.json"
!python scripts/summarize_walk_forward.py --experiment E22 {WF} --out "$OUT_DIR/batch3/e22_summary_shared_epoch.json"

In [ ]:
# Summary 4: E22 against E20 (batch-2 parent) and E05 (E03, RevIN on), window by window
def load(p): return {f['fold']: f for f in json.load(open(p))['folds']}
e22p, e22s = load(f'{B3}/e22_summary_per_horizon.json'), load(f'{B3}/e22_summary_shared_epoch.json')
e20p, e05 = load(f'{OUT_DIR}/batch2/e20_summary_per_horizon.json'), load(f'{OUT_DIR}/e05/summary_e05.json')
rows = []
print(f"{'window':8s} {'h':4s} {'E22':>6s} {'E20':>6s} {'E05':>6s} {'E22-E20':>8s} {'E22-E05':>8s} {'floor':>6s} {'XGB':>6s}")
for w in [f for f, _, _ in FOLDS] + ['official']:
    for h in ('h1', 'h5', 'h10'):
        a, b = e22p[w]['h'][h], e20p[w]['h'][h]
        c = e05[w]['h'][h]['model_mean'] if w in e05 else None
        fl = max(a['trend_continues'], a['trend_reverses'])
        rows.append({'window': w, 'h': h, 'e22': a['model_mean'], 'e20': b['model_mean'], 'e05': c, 'floor': fl, 'xgb': a['xgb']})
        e05s = 'n/a' if c is None else f"{c:.3f}"
        d05 = 'n/a' if c is None else f"{a['model_mean'] - c:+.3f}"
        print(f"{w:8s} {h:4s} {a['model_mean']:6.3f} {b['model_mean']:6.3f} {e05s:>6s} "
              f"{a['model_mean'] - b['model_mean']:+8.3f} {d05:>8s} {fl:6.3f} {a['xgb']:6.3f}")
json.dump({'rows': rows, 'note': 'E22 and E20 per-horizon best epoch; E05 shared epoch (predates per-horizon scoring)'},
          open(f'{B3}/e22_vs_e20_e05.json', 'w'), indent=2)
print('wrote', f'{B3}/e22_vs_e20_e05.json')